## fMRI time series to covariance matrices


This notebook shows how raw fMRI time series are preprocessed into Gaussian-copula covariance matrices -- the core data representation used throughout the analysis.

Raw time series must be downloaded from Zenodo before running this notebook:
- **MA dataset** (Uhrig et al. 2018): https://zenodo.org/records/10572216 -- file `CoCoMac/timeseries.npy`
- **DBS dataset** (Tasserie et al. 2022): contact corresponding authors

Each state array is expected to have shape `(N_subjects, T_samples, 82_regions)`.

The output is `data/covariance_matrices_gc.h5`, a nested HDF5 file with one group per dataset and one dataset per brain state.


Note: This notebook should be run from the `high-order-anesthesia` folder to ensure the correct imports and file paths are used.


In [11]:
from pathlib import Path
import os
def ensure_project_root(target_name: str = "high-order-anesthesia") -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == target_name:
        return cwd
    for parent in cwd.parents:
        if parent.name == target_name:
            os.chdir(parent)
            return parent
    raise RuntimeError(
        f"Could not find '{target_name}' in current path or parents. "
        f"Please run the notebook from inside the project."
    )
ROOT = ensure_project_root("high-order-anesthesia")
print(f"Now in: {ROOT.name}")


Now in: high-order-anesthesia


In [12]:
import numpy as np
from src.hoi_anesthesia.preprocessing import generate_covmats # to generate dict
from src.hoi_anesthesia.io import load_covariance_dict # to load dict


In [13]:
data_path = "data"

# Load timeseries data (156 scans, 500 timepoints, 82 regions)
ts = np.load(f"{data_path}/timeseries.npy", allow_pickle=False)  # shape (156, 500, 82)
print(f"timeseries shape: {ts.shape}")

# Load participants metadata to map scans to conditions
import pandas as pd
participants = pd.read_csv(f"{data_path}/participants.tsv", delimiter="\t")
print(f"Total scans in metadata: {len(participants)}")
print(f"\nScans per condition:")
print(participants["cond"].value_counts().sort_index())

# Map condition names from file to expected keys
condition_mapping = {
    "awake":                 "MA_awake",
    "ketamine":              "ketamine",
    "moderate-propofol":     "moderate_propofol",
    "deep-propofol":         "deep_propofol",
    "moderate-sevoflurane":  "ts_selv2",
    "deep-sevoflurane":      "ts_selv4",
}

# Extract scans by condition
all_states = {"MA": {}}
for file_condition, expected_key in condition_mapping.items():
    # Get indices for this condition
    idx = participants[participants["cond"] == file_condition].index.tolist()
    # Extract timeseries slices
    scans = ts[idx]  # shape (n_scans, 500, 82)
    all_states["MA"][expected_key] = scans
    print(f"{expected_key:20s} <- {file_condition:20s}  shape: {scans.shape}")

print(f"\nBuilding covariance matrices...")
generate_covmats(all_states, data_path, covmat_name="example_covariance_matrices_gc")


timeseries shape: (156, 500, 82)
Total scans in metadata: 156

Scans per condition:
cond
awake                   31
deep-propofol           30
deep-sevoflurane        20
ketamine                25
moderate-propofol       25
moderate-sevoflurane    25
Name: count, dtype: int64
MA_awake             <- awake                 shape: (31, 500, 82)
ketamine             <- ketamine              shape: (25, 500, 82)
moderate_propofol    <- moderate-propofol     shape: (25, 500, 82)
deep_propofol        <- deep-propofol         shape: (30, 500, 82)
ts_selv2             <- moderate-sevoflurane  shape: (25, 500, 82)
ts_selv4             <- deep-sevoflurane      shape: (20, 500, 82)

Building covariance matrices...
MA/MA_awake - (31, 500, 82)  →  torch.Size([31, 82, 82])
MA/ketamine - (25, 500, 82)  →  torch.Size([25, 82, 82])
MA/moderate_propofol - (25, 500, 82)  →  torch.Size([25, 82, 82])
MA/deep_propofol - (30, 500, 82)  →  torch.Size([30, 82, 82])
MA/ts_selv2 - (25, 500, 82)  →  torch.Size([25

Inspect resulting dictionary


In [24]:
example_covmats = load_covariance_dict(data_path + '/example_covariance_matrices_gc.h5')

# Print all datasets (should be 'MA')
print("Datasets in file:")
datasets = example_covmats.keys()
print(list(datasets))

# Print all states within MA dataset
print("\nStates in MA dataset:")
ma_states = example_covmats['MA'].keys()
print(list(ma_states))

# Print shape of deep_propofol covariance matrices
print("\nShape of deep_propofol covariance matrices:")
deep_propofol_shape = example_covmats['MA']['deep_propofol'].shape
print(f"  {deep_propofol_shape}  (n_subjects, 82_regions, 82_regions)")


Datasets in file:
['MA']

States in MA dataset:
['MA_awake', 'deep_propofol', 'ketamine', 'moderate_propofol', 'ts_selv2', 'ts_selv4']

Shape of deep_propofol covariance matrices:
  (30, 82, 82)  (n_subjects, 82_regions, 82_regions)


Below is the same inspection applied to the main `covariance_matrices_gc.h5` file (which includes both MA and DBS datasets):


In [27]:
all_covmats = load_covariance_dict(data_path + '/covariance_matrices_gc.h5')

# Print all datasets (should be 'MA')
print("Datasets in file:")
datasets = all_covmats.keys()
print(list(datasets))

print("\nStates in MA dataset:")
ma_states = all_covmats['MA'].keys()
print(list(ma_states))
print("\nStates in DBS dataset:")
dbs_states = all_covmats['DBS'].keys()
print(list(dbs_states))

print("\nShape of deep_propofol covariance matrices:")
deep_propofol_shape = all_covmats['MA']['deep_propofol'].shape
print(f"  {deep_propofol_shape}  (n_subjects, 82_regions, 82_regions)")
print("\nShape of ts_on_3V covariance matrices:")
ts_on_3V_shape = all_covmats['DBS']['ts_on_3V'].shape
print(f"  {ts_on_3V_shape}  (n_subjects, 82_regions, 82_regions)")


Datasets in file:
['DBS', 'MA']

States in MA dataset:
['MA_awake', 'deep_propofol', 'ketamine', 'moderate_propofol', 'ts_selv2', 'ts_selv4']

States in DBS dataset:
['DBS_awake', 'ts_off', 'ts_on_3V', 'ts_on_3V_control', 'ts_on_5V', 'ts_on_5V_control']

Shape of deep_propofol covariance matrices:
  (23, 82, 82)  (n_subjects, 82_regions, 82_regions)

Shape of ts_on_3V covariance matrices:
  (31, 82, 82)  (n_subjects, 82_regions, 82_regions)
